# OpenPlaque — BACCE-style deterministic RCA tracker

Self-contained notebook. No `%run`. Starts from `main`.

Goal: use BACCE's 500-direction action lattice and forward-direction logic, but replace missing learned DDQN Q-values with deterministic image support (HU + vesselness + smoothness + distance-from-aorta) so we can test whether BACCE-style local stepping improves RCA extension beyond the already validated proximal path.


In [ ]:
# FIRST EXECUTABLE CELL
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!rm -rf /content/OpenPlaque /content/BACCE
!git clone -q --branch bacce-deterministic-tracker-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!git clone -q https://github.com/514sz/Branch-aware-centerline-extraction.git /content/BACCE
%pip -q install pydicom SimpleITK scipy scikit-image matplotlib pandas torch


In [ ]:
import sys, shutil, math
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, SimpleITK as sitk
from scipy import ndimage as ndi
from skimage.filters import frangi
from skimage.graph import route_through_array

sys.path.insert(0,'/content/OpenPlaque/src')
sys.path.insert(0,'/content/BACCE')
from openplaque.study import OpenPlaqueStudy
from utils import create_actions

ROOT=Path('/content/drive/MyDrive/OpenPlaque')
OUT=ROOT/'BACCE_Deterministic_Tracker'; OUT.mkdir(parents=True,exist_ok=True)


## 1. Load series 7, cached TotalSegmentator aorta, and current BACCE seed

In [ ]:
dz=ROOT/'Full_DICOM.zip'; lz=Path('/content/Full_DICOM.zip')
if not lz.exists() or lz.stat().st_size!=dz.stat().st_size: shutil.copyfile(dz,lz)
shutil.rmtree('/content/full_dicom_bacce_det',ignore_errors=True)
study=OpenPlaqueStudy(str(lz),extract_root='/content/full_dicom_bacce_det')
img,ct,_=study.load_series(7); ct=np.asarray(ct)
sp_xyz=np.array(img.GetSpacing(),float); sp_zyx=sp_xyz[::-1]

ap=ROOT/'RCA_Ostium_TotalSegmentator'/'aorta_series7_totalseg.nii.gz'
if not ap.exists(): raise FileNotFoundError(ap)
aim=sitk.ReadImage(str(ap))
if aim.GetSize()!=img.GetSize() or not np.allclose(aim.GetSpacing(),img.GetSpacing()):
    aim=sitk.Resample(aim,img,sitk.Transform(),sitk.sitkNearestNeighbor,0,sitk.sitkUInt8)
aorta=sitk.GetArrayFromImage(aim)>0
dist_aorta=ndi.distance_transform_edt(~aorta,sampling=sp_zyx)
print('shape:',ct.shape,' spacing xyz:',sp_xyz)


## 2. Recover the ostium candidate and proximal image-supported route

In [ ]:
# Use the same compatibility-run ostium logic, then reconstruct a proximal route.
zlo,zhi=225,305
band=np.zeros_like(aorta,bool); band[zlo:zhi+1]=True
a_hu=ct[aorta & band]
blood_thr=float(np.clip(np.median(a_hu)*0.43,180,360))
shell=(dist_aorta>0.8)&(dist_aorta<=8.0)&band
cand=(ct>=blood_thr)&shell
cand=ndi.binary_closing(cand,structure=np.ones((3,3,3),bool),iterations=1)
lab,n=ndi.label(cand,structure=np.ones((3,3,3),bool))
rows=[]
for k in range(1,n+1):
    c=np.argwhere(lab==k)
    if len(c)<5: continue
    dv=dist_aorta[tuple(c.T)]
    if dv.min()>3 or dv.max()<2: continue
    mm=c.astype(float)*sp_zyx
    if len(mm)<3: continue
    ev=np.sort(np.linalg.eigvalsh(np.cov(mm,rowvar=False)))[::-1]
    L=float(np.sqrt(max(1e-6,12*ev[0])))
    z0,y0,x0=c.min(0); z1,y1,x1=c.max(0)+1
    sub=(lab[z0:z1,y0:y1,x0:x1]==k)
    rmax=float(ndi.distance_transform_edt(sub,sampling=sp_zyx).max())
    if not (2<=L<=20 and rmax<=4.5): continue
    q=c[np.argmin(dv)]
    rows.append((k,L,rmax,float(dv.min()),float(dv.max()),q))
if not rows: raise RuntimeError('No ostium-like candidate')
rows=sorted(rows,key=lambda t:(t[1]+2*t[4]-2*t[2]),reverse=True)
k,L,rmax,mind,maxd,ostium=rows[0]
ostium=np.array(ostium,float)
print('ostium zyx:',ostium,' candidate L:',L,' r:',rmax)

# Local ROI and support
pad=np.array([24,80,80]); lo=np.maximum(np.floor(ostium-pad).astype(int),0); hi=np.minimum(np.ceil(ostium+pad).astype(int)+1,np.array(ct.shape))
roi=ct[lo[0]:hi[0],lo[1]:hi[1],lo[2]:hi[2]].astype(float)
ar=aorta[lo[0]:hi[0],lo[1]:hi[1],lo[2]:hi[2]]
sm=ndi.gaussian_filter(roi,0.7)
v=frangi(sm, sigmas=[0.7,1.0,1.4,1.8], black_ridges=False)
v99=np.percentile(v[v>0],99) if np.any(v>0) else 1.0
vn=np.clip(v/max(v99,1e-6),0,1)
med_hu=float(np.median(ct[aorta & band]))
lohu=max(140.,blood_thr*.55); hihu=max(lohu+100.,med_hu*1.35)
ints=np.clip((sm-lohu)/(hihu-lohu),0,1)
support=.58*ints+.42*vn
da=ndi.distance_transform_edt(~ar,sampling=sp_zyx)
cost=1/(.06+support); cost[ar]+=40; cost[da<.6]+=15; cost[sm<lohu]+=12

start=np.round(ostium-lo).astype(int)
# Candidate endpoints 16-28 mm away and progressively farther from aorta.
pts=np.argwhere((support>.40)&(da>6)&(da<26))
eps=[]
for p in pts[::max(1,len(pts)//2500)]:
    d=np.linalg.norm((p-start)*sp_zyx)
    if 16<=d<=30: eps.append((float(support[tuple(p)]),*map(int,p)))
eps=sorted(eps,reverse=True)[:120]
routes=[]
for s,ez,ey,ex in eps:
    try: path,_=route_through_array(cost,tuple(start),(ez,ey,ex),fully_connected=True,geometric=True)
    except Exception: continue
    p=np.asarray(path,int); step=np.diff(p.astype(float),axis=0)*sp_zyx; Lp=float(np.linalg.norm(step,axis=1).sum())
    if Lp<16: continue
    dd=da[tuple(p.T)]; su=support[tuple(p.T)]
    outward=float(np.mean(np.diff(dd)>=-.35)) if len(dd)>1 else 0
    score=0.45*np.mean(su)+0.25*outward+0.20*np.tanh(dd[-1]/12)+0.10*np.tanh(Lp/20)
    routes.append((score,Lp,p))
if not routes: raise RuntimeError('Could not reconstruct proximal route')
routes.sort(key=lambda t:t[0],reverse=True)
route=routes[0][2]
route_global=route+lo
seg=np.linalg.norm(np.diff(route_global.astype(float),axis=0)*sp_zyx,axis=1)
arc=np.r_[0,np.cumsum(seg)]
seed_idx=int(np.argmin(np.abs(arc-6.0)))
seed=route_global[seed_idx].astype(float)
print('route length mm:',arc[-1],' seed at mm:',arc[seed_idx],' seed:',seed)


## 3. Build BACCE-style 500-direction deterministic action lattice

In [ ]:
actions=create_actions(500,dis=1.5,spacing=sp_xyz).astype(float)
# Remove zero offsets caused by voxel rounding.
actions=np.unique(actions,axis=0)
actions=actions[np.linalg.norm(actions*sp_zyx,axis=1)>0]
print('unique nonzero actions:',len(actions))

# Direction estimated from route between ~3 and 9 mm.
i0=int(np.argmin(np.abs(arc-3.0))); i1=int(np.argmin(np.abs(arc-9.0)))
prev_dir=(route_global[i1]-route_global[i0]).astype(float)
prev_dir=prev_dir/(np.linalg.norm(prev_dir*sp_zyx)+1e-9)
print('initial route direction zyx:',prev_dir)


## 4. Deterministic BACCE-style stepper

In [ ]:
def sample_trilinear(arr,p):
    return float(ndi.map_coordinates(arr,np.asarray(p,float)[:,None],order=1,mode='nearest')[0])

def angle_deg(a,b):
    aa=a*sp_zyx; bb=b*sp_zyx
    na=np.linalg.norm(aa); nb=np.linalg.norm(bb)
    if na<1e-8 or nb<1e-8: return 180.
    return float(np.degrees(np.arccos(np.clip(np.dot(aa,bb)/(na*nb),-1,1))))

def local_score(p,prev,a):
    # p is global zyx candidate endpoint
    if np.any(p<1) or np.any(p>=np.array(ct.shape)-1): return -1e9,{}
    hu=sample_trilinear(ct,p)
    # vesselness computed on full-volume local approximation: use intensity + local Hessian proxy from ROI support where available
    d=sample_trilinear(dist_aorta,p)
    ang=angle_deg(a,prev)
    if ang>80: return -1e9,{}
    # local tubular support from a 5x5x5 neighborhood
    q=np.round(p).astype(int); z,y,x=q
    z0,z1=max(0,z-2),min(ct.shape[0],z+3); y0,y1=max(0,y-2),min(ct.shape[1],y+3); x0,x1=max(0,x-2),min(ct.shape[2],x+3)
    sub=ct[z0:z1,y0:y1,x0:x1].astype(float)
    loc_med=float(np.median(sub)); bright=np.clip((hu-lohu)/(hihu-lohu),0,1)
    neigh=np.clip((loc_med-lohu)/(hihu-lohu),0,1)
    forward=max(0,math.cos(math.radians(ang)))
    # encourage outward progress but don't force monotonicity too aggressively after proximal segment
    score=0.42*bright+0.18*neigh+0.22*forward+0.12*np.tanh(d/8)+0.06*np.tanh(max(0,hu-150)/300)
    return score,{'hu':hu,'dist_aorta':d,'angle':ang,'bright':bright,'neigh':neigh}

max_steps=55
cur=seed.copy(); prev=prev_dir.copy(); path=[cur.copy()]; log=[]
for t in range(max_steps):
    scored=[]
    for a in actions:
        p=cur+a
        s,m=local_score(p,prev,a)
        if s>-1e8: scored.append((s,p,a,m))
    if not scored: break
    scored.sort(key=lambda x:x[0],reverse=True)
    best=scored[0]
    if best[0]<0.43: break
    _,p,a,m=best
    cur=p; prev=a/(np.linalg.norm(a*sp_zyx)+1e-9)
    path.append(cur.copy()); log.append({'step':t+1,'score':best[0],**m,'z':cur[0],'y':cur[1],'x':cur[2]})
path=np.asarray(path,float)
seg2=np.linalg.norm(np.diff(path,axis=0)*sp_zyx,axis=1) if len(path)>1 else np.array([])
arc2=np.r_[0,np.cumsum(seg2)]
print('deterministic extension length mm:',arc2[-1] if len(arc2) else 0,' steps:',len(path)-1)
display(pd.DataFrame(log).head(12))


## 5. Compare deterministic tracker against known proximal route and visualize

In [ ]:
# nearest physical distance from each deterministic point to known proximal route
known=route_global.astype(float)
errs=[]
for p in path:
    errs.append(float(np.min(np.linalg.norm((known-p)*sp_zyx,axis=1))))
errs=np.array(errs)
print('agreement first 15 mm: median/max mm =',np.median(errs[arc2<=15]) if np.any(arc2<=15) else np.nan, np.max(errs[arc2<=15]) if np.any(arc2<=15) else np.nan)

z=int(round(seed[0])); y=int(round(seed[1])); x=int(round(seed[2]))
fig,axs=plt.subplots(1,4,figsize=(20,5))
axs[0].imshow(ct[z],cmap='gray',vmin=-200,vmax=900)
axs[0].contour(aorta[z].astype(float),levels=[.5],linewidths=1.2)
axs[0].plot(route_global[:,2],route_global[:,1],'.',ms=2,label='known proximal')
axs[0].plot(path[:,2],path[:,1],'-',lw=1.5,label='BACCE-style')
axs[0].plot(x,y,'o',ms=6,label='seed')
axs[0].legend(fontsize=8); axs[0].set_title(f'axial z={z}'); axs[0].axis('off')

r=55; y0=max(0,y-r);y1=min(ct.shape[1],y+r);x0=max(0,x-r);x1=min(ct.shape[2],x+r)
axs[1].imshow(ct[z,y0:y1,x0:x1],cmap='gray',vmin=-200,vmax=900)
axs[1].plot(route_global[:,2]-x0,route_global[:,1]-y0,'.',ms=2)
axs[1].plot(path[:,2]-x0,path[:,1]-y0,'-',lw=1.5)
axs[1].set_title('seed neighborhood'); axs[1].axis('off')

axs[2].plot(arc2,errs)
axs[2].axhline(2.0,ls='--')
axs[2].set_xlabel('tracker arc length (mm)'); axs[2].set_ylabel('distance to known route (mm)'); axs[2].set_title('agreement QC')

df=pd.DataFrame(log)
if len(df): axs[3].plot(df.step,df.score,label='step score'); axs[3].plot(df.step,df.dist_aorta,label='dist aorta')
axs[3].legend(fontsize=8); axs[3].set_title('step QC')
plt.tight_layout(); p=OUT/'bacce_deterministic_tracker_report.png'; fig.savefig(p,dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig)

summary=pd.DataFrame([{'known_route_length_mm':float(arc[-1]),'seed_arc_mm':float(arc[seed_idx]),'tracker_extension_mm':float(arc2[-1]) if len(arc2) else 0.0,'tracker_steps':int(len(path)-1),'median_error_first15_mm':float(np.median(errs[arc2<=15])) if np.any(arc2<=15) else np.nan,'max_error_first15_mm':float(np.max(errs[arc2<=15])) if np.any(arc2<=15) else np.nan,'final_dist_aorta_mm':float(sample_trilinear(dist_aorta,path[-1])) if len(path) else np.nan}])
display(summary); summary.to_csv(OUT/'bacce_deterministic_tracker_summary.csv',index=False)
print('Saved:',p)
